# Week 3 — The Retrieval Lab

Week 2 ended with one honest gap. Your context template had a `sources=` block, and
you filled it by hand — which only works if you already know which passage answers the
question. This session builds the thing that fills it, and then closes the loop by
handing the result back to the model.

**How to work through this.** Run the cells in order, on **your own corpus**. Every
number you get should come from material you chose, not from the example below.

**On cost, and this one bites.** Embeddings are rate-limited to roughly **100 texts per
minute**, counted per text rather than per request — batching cannot speed it up, only
waiting can. The sample corpus below is small enough to embed in the notebook. **Your
real corpus is not:** embed it once with `python ingest.py`, which paces itself and
writes to `./chroma/`, and it stays embedded across every kernel restart afterwards.

Generation, in Parts 6 and 7, is the separate **6 calls per minute** limit.

**Before you start:** you need your Week 2 context template — `SYSTEM`, `TASK`,
`FORMAT`. If you did not keep it, reconstruct it now; Part 6 depends on it.

In [6]:
# Run this notebook from any folder. Week 3's modules live in week3/, and this
# lab also reuses Week 2's context_lab, so both -- plus common/ -- go on the path.
import sys, pathlib
_root = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (p / "requirements.txt").exists()), pathlib.Path.cwd().parent)
for _d in ("common", "week2", "week3"):
    sys.path.insert(0, str(_root / _d))

In [7]:
from chunking import chunk_text
from embedding_client import EmbeddingClient
from vector_store import VectorStore

emb = EmbeddingClient()
print(f"embeddings: {emb.provider} / {emb.model}")

embeddings: gemini / gemini-embedding-2


---
## Part 1 — Your corpus, in pieces

Replace `MY_CORPUS` with real text from the domain you proposed in Week 1. Paste two or
three documents' worth — a few hundred words is enough to make the chunking visible,
and you will point this at the real thing for Lab 1.

In [ ]:
MY_CORPUS = """
Модель може працювати лише з тим, що є в її контексті. Усе, що потрапляє у вікно
контексту, оплачується під час кожного виклику. Пошук потрібен для того, щоб
програма знаходила потрібний фрагмент, а розробник не мусив заздалегідь знати,
який саме абзац відповідає на запитання.

Lab 2 - Knowledge graph and Graph RAG - due end of Week 7 - 25% of the final grade.
Deliverables: Neo4j database and graph model; re-seedable ingest.py; Cypher queries;
graph retrieval implementation; measured comparison against the Lab 1 system on the
shared gold set; README with AI-use disclosure.

Chroma's default distance is not cosine. An unconfigured collection uses squared
Euclidean distance, so a collection must be created with hnsw:space set to cosine
to match the metric this course teaches.
"""

chunks = chunk_text(MY_CORPUS, chunk_size=60, overlap=15)
print(f"{len(MY_CORPUS.split())} words -> {len(chunks)} chunks\n")
for i, c in enumerate(chunks):
    print(f"[{i}] {c[:90]}...")

Now change `chunk_size` and `overlap` and run it again. Watch what happens to a
sentence that sits on a boundary.

**Write down the size you settle on and why.** There is no correct answer — too small
loses the context that gave a chunk its meaning, too large dilutes relevance with
unrelated material. You will defend this choice in Lab 1.

### ✍️ Your notes

**Chunk size and overlap I chose:**

**Why:**

---
## Part 2 — Look at an actual embedding

The lecture claimed two things about the vector. Check both yourself rather than
taking them on faith.

In [ ]:
short = emb.embed("контекст")
long_ = emb.embed(" ".join(chunks))

print(f"1 word   -> {len(short)} numbers")
print(f"{len(' '.join(chunks).split()):>3} words -> {len(long_)} numbers")
print(f"\nsame length? {len(short) == len(long_)}")
print(f"\nfirst 8 numbers: {[round(x, 4) for x in short[:8]]}")

**Claim 1: fixed length regardless of input.** Confirmed above.

**Claim 2: nearby means related.** The numbers themselves are unreadable — what makes
them useful is how they compare. Pick a question about your corpus, and a sentence that
has nothing to do with it.

In [ ]:
QUESTION = "Що оплачується під час кожного виклику?"
UNRELATED = "Рецепт хліба починається з борошна та води."

q = emb.embed(QUESTION, task_type="RETRIEVAL_QUERY")
relevant = emb.embed(chunks[0], task_type="RETRIEVAL_DOCUMENT")
unrelated = emb.embed(UNRELATED, task_type="RETRIEVAL_DOCUMENT")

print(f"cosine(relevant chunk, question)  = {EmbeddingClient.cosine(relevant, q):.4f}")
print(f"cosine(unrelated text, question)  = {EmbeddingClient.cosine(unrelated, q):.4f}")

The unrelated score is probably **not** near zero — two arbitrary sentences in the same
language share a lot of structure. That is fine. Retrieval never needs an absolute
threshold; it needs the relevant one to rank **higher**, which is what you just checked.

---
## Part 3 — Retrieval, end to end

`VectorStore` is the ingest-once, query-every-call split from the lecture. `add()` is
ingestion. `search()` runs per question.

Two things about it are deliberate. The collection is created with
`hnsw:space = "cosine"` — Chroma's default is squared Euclidean, which is **not** what
this course measures. And the store is **persistent**: it writes to `./chroma/` and
survives a kernel restart, which is what makes a rate-limited ingest survivable.

The sample below is a handful of chunks, so embedding it here is fine.

In [ ]:
store = VectorStore(embedder=emb, name="lab_demo", reset=True)
# Store WHERE each chunk came from, so retrieval can cite it later. Here the
# corpus is one block of text, so we label it "my_corpus"; ingest.py records the
# real filename for each document instead.
metadatas = [{"source": "my_corpus", "chunk_index": j} for j in range(len(chunks))]
store.add(chunks, metadatas=metadatas)
print(f"{store.count()} chunks indexed\n")

for row in store.search(QUESTION, n_results=3):
    src = row["metadata"].get("source", "?")
    print(f"distance={row['distance']:.4f}  [{src}]  {row['chunk'][:70]}...")

Every result now carries a `metadata` field saying which document it came from — that is
the provenance you will cite in Part 6. On your real corpus, `ingest.py` fills `source`
with the actual filename and `chunk_index`/`start_word` with the position inside it.

`search()` shows what is *relevant*. To see what is actually stored — which is how you
check a chunk boundary — look directly:

In [8]:
print(f"{store.count()} chunks in {store.collections()}\n")
for row in store.peek(3):
    print(f"[{row['id']}] ({row['metadata'].get('source', '?')}) {row['chunk'][:90]}...")

177 chunks in ['chunks', 'lab_demo']

[0] (week-01-lecture-plan.md) # Modern AI Systems **Session:** Week 1, session 1 of 2 (lecture) **Duration:** 90 minutes...
[1] (week-01-lecture-plan.md) — nothing in Week 1 is graded — the Milestone 0 proposal it sets up is due end of Week 2. ...
[2] (week-01-lecture-plan.md) Notes | |---|---|---| | 0–10 | **Framing.** The central question; "we are not building ano...


Read chunk `[0]` against your source text. Did the boundary land mid-sentence? That is the
trade-off from Part 1, now visible in what you actually stored rather than argued about on
a slide.

From a terminal you can browse the whole collection instead:
`chroma browse lab_demo --path ./chroma` — arrows to move, `Return` to expand, `q` to quit.

**For your own corpus, stop and do this in a terminal instead:**

```bash
python ingest.py corpus/ --chunk-size 200 --overlap 40
```

Then come back and open what it built — no re-embedding, however often you restart:

```python
store = VectorStore(name="chunks")     # already populated
print(store.count())
```

Re-run `ingest.py --reset` only when you change the chunk size, since that invalidates
every boundary already stored.

`distance` here is **1 − cosine similarity**, so smaller is closer.

Try three or four more questions about your own corpus. Look for one it gets right and
one it gets wrong — you need both for Part 6.

### ✍️ Your notes

**A question it answered well:**

**A question it got wrong, and what it returned instead:**

---
## Part 4 — The gold set (due this week)

Ten representative queries for your domain, each with something checkable: a
distinctive phrase that the *correct* chunk must contain.

**Write these before you tune anything.** A gold set written afterwards, to fit results
you already have, measures nothing. This is the seed every later improvement in the
course is compared against — including Week 4, next session.

In [ ]:
GOLD = [
    {"query": "Що оплачується під час кожного виклику?", "expect": "оплачується"},
    {"query": "What is Lab 2 worth?", "expect": "25%"},
    {"query": "Which distance does Chroma use by default?", "expect": "Euclidean"},
    # ... write seven more, for YOUR corpus.
]

print(f"{len(GOLD)} queries written. Target: 10.")

In [ ]:
def hits_at_k(store, gold, k=5):
    """How many gold queries return the right chunk in their top k.

    A rough count, not yet a measurement -- Week 4 turns this into recall@k
    properly, across several values of k and with the failures diagnosed.
    """
    found = 0
    for item in gold:
        results = store.search(item["query"], n_results=k)
        if any(item["expect"].lower() in r["chunk"].lower() for r in results):
            found += 1
    return found


print(f"{emb.model}: {hits_at_k(store, GOLD)} / {len(GOLD)} found in top 5")

In [ ]:
alt = EmbeddingClient(provider="openrouter")
alt_store = VectorStore(embedder=alt, name="lab_demo_alt", reset=True)
alt_store.add(chunks)

print(f"{emb.model:<24} {hits_at_k(store, GOLD)} / {len(GOLD)}")
print(f"{alt.model:<24} {hits_at_k(alt_store, GOLD)} / {len(GOLD)}")

**Discuss whichever way it lands.** A tie is the most likely result on a small corpus,
and a tie is itself an answer: it means the choice should be made on the things that are
not retrieval quality — rate limit, cost, dimensionality and therefore storage, and
whether `task_type` works at all.

That last one is not a tiebreaker, it is decisive. Query/document asymmetry works on the
Gemini backend and is a confirmed **no-op** through OpenRouter, even routing the very
same model. Check `embedding_client.py`'s capability matrix before you choose.

### ✍️ Your notes

**Which model I will use for Lab 1, and the number that decided it:**

---
## Part 6 — Hand the chunks to the model

This is the loop closing. Everything below is your Week 2 template — the only new line
is the one that puts retrieved text where pasted text used to go.

In [9]:
from context_lab import ask

SYSTEM = (
    "You answer questions using only the sources you are given. If the sources do "
    "not answer the question, reply exactly: not stated. Never fill a gap from your "
    "own knowledge."
)
TASK = "Answer the question using only the SOURCES above."
FORMAT = "Two sentences maximum. Put the source number in square brackets after each claim."

def rag(question, *, k=3, show_prompt=True):
    """Retrieve, assemble, generate -- then print where each source came from."""
    rows = store.search(question, n_results=k)
    result = ask(question, system=SYSTEM, task=TASK, fmt=FORMAT,
                 sources=[r["chunk"] for r in rows], show_prompt=show_prompt,
                 label=f"rag k={k}")
    print("\nSources (what the [n] citations point to):")
    for i, r in enumerate(rows, 1):
        m = r["metadata"]
        where = f"chunk {m.get('chunk_index', '?')}"
        if "start_word" in m:
            where += f", ~word {m['start_word']}"
        print(f"  [{i}] {m.get('source', 'unknown')} ({where})")
    return result

QUESTION = "What do we learn in week 3?"

result = rag(QUESTION)


=== rag k=3 =====================================================
SYSTEM (sent as a system instruction, not as prompt text)
You answer questions using only the sources you are given. If the sources do not answer the question, reply exactly: not stated. Never fill a gap from your own knowledge.

SOURCES
[1] # Week 3 · Lecture — Embeddings, semantic search, and RAG **Session:** Week 3, session 1 of 2 (lecture) **Duration:** 90 minutes **Syllabus reference:** `Modern_AI_Systems_Syllabus_v2.md` §6 Week 3 **Guiding question:** How can software find information by meaning, and hand what it finds to a model? **Organising principle:** Week 2 closed by naming the limitation its own prompt skeleton could not remove: the sources block was still pasted in by hand, which required the developer to already know which passage answered the question. This session removes that assumption and then completes the loop. It is not new material bolted onto the template; it is the automation of the one block o

Read the printed context before you read the answer. The `SOURCES` block was assembled
by your code, from your corpus, moments ago — nothing else in the template changed
since Week 2.

The **Sources** list under the answer is the provenance: it maps each `[1]`, `[2]` the
model cited back to the document it was retrieved from, and where in that document. That
is what turns "a grounded answer" into "a grounded answer you can check" — the thing
Lab 1 asks for.

Now run the question Part 3 got **wrong**. Retrieval failing means the model is
grounded in the wrong passage, and a wrong passage does not announce itself.

In [ ]:
# The question your retrieval handled badly. Read the cited chunk, not just the answer.
BAD = "..."   # replace with your own

# result = rag(BAD)

**Find at least one case where the answer reads well and the cited chunk does not
support it.** That case is the entire reason Week 4 exists, and you will need it next
session.

### ✍️ Your notes

**A fluent answer whose source did not support it:**

**Where it went wrong — chunking, retrieval, or generation?**

---
## Part 7 — The decisions that are yours

The lecture listed four: how many chunks, in what order, truncated how, numbered how.
Here is what the first one costs.

In [ ]:
from context_lab import counttokens, render

for k in (1, 3, 5, 10):
    retrieved = [row["chunk"] for row in store.search(QUESTION, n_results=k)]
    prompt = render(QUESTION, task=TASK, fmt=FORMAT, sources=retrieved)
    print(f"  k={k:>2}  ->  {counttokens(prompt):>5} prompt tokens, every call")

Every one of those tokens is billed on **every** call — the Week 2 arithmetic, arriving
somewhere you did not expect it. Retrieving ten chunks where three would do is a cost
decision you make by accident unless you look.

Run `rag(QUESTION, k=1)` and `rag(QUESTION, k=10)` and compare the answers, not just the
prices. More context is not reliably better: the relevant passage now competes with
material that is merely nearby.

**The other three decisions** are yours to explore in Lab 1. Ordering: `search()` returns
nearest first — is that the best position for the strongest chunk? Truncation: what do
you cut when the budget is exceeded? Numbering: `render()` writes `[1]`, `[2]`, which is
the only reason a citation can be checked at all.

### ✍️ Your notes

**The k I chose, and what it cost:**

---
## Before you close this

**Due this week:** your ten-query gold set, and a program that takes a question and
returns a grounded answer from your own corpus.

**What you still do not know:** how often it is right. You have checked a handful of
queries by eye and measured one choice against the gold set. That is not the same as
knowing the system works, and the fluent-but-unsupported answer you found in Part 6 is
the proof that reading outputs will not settle it.

**Week 4 measures it** — recall@k across the whole gold set, the failure modes behind
each miss, and how to tell which stage went wrong.

In [ ]:
from context_lab import calls
print(f"generation calls this session: {calls()}")
print(f"embedding requests:            {emb.calls}")
print(f"texts embedded:                {emb.texts_embedded}  (limit ~100/minute)")